
# NBA Player `personId` → Full Name CSV (2013–14 to 2024–25)

This notebook fetches NBA players for each season from **2013–14** through **2024–25** using `nba_api` and saves a CSV with two columns:

- `personId`
- `playerFullName`

## How to use
1. Run the **Install dependencies** cell.
2. Run the **Build CSV** cell.
3. The CSV will be saved as `nba_players_2013_2025.csv` in the current working directory.


In [ ]:

# Install dependencies (uncomment if needed)
# If you're running in an environment where nba_api/pandas/tqdm are not installed,
# remove the leading '#' characters and run this cell.
# !pip install nba_api pandas tqdm


In [ ]:

import time
import pandas as pd
from tqdm import tqdm

# nba_api imports
from nba_api.stats.endpoints import commonallplayers

def season_str(year_start: int) -> str:
    """Return NBA-style season string like '2013-14' for a given start year 2013."""
    return f"{year_start}-{str(year_start + 1)[-2:]}"

# Seasons from 2013-14 to 2024-25 (inclusive)
seasons = [season_str(y) for y in range(2013, 2025)]

all_dfs = []

for s in tqdm(seasons, desc="Fetching players per season"):
    # We'll retry a few times in case the stats API is flaky
    for attempt in range(5):
        try:
            # is_only_current_season=1 => returns players active for that season
            resp = commonallplayers.CommonAllPlayers(
                league_id='00',
                season=s,
                is_only_current_season=1
            )
            df = resp.get_data_frames()[0].copy()
            # Keep only the columns we need
            df = df[["PERSON_ID", "DISPLAY_FIRST_LAST"]].rename(
                columns={"PERSON_ID": "personId", "DISPLAY_FIRST_LAST": "playerFullName"}
            )
            df["SEASON"] = s
            all_dfs.append(df)
            # Be respectful to the API
            time.sleep(1.2)
            break
        except Exception as e:
            wait = 2 ** attempt
            print(f"[{s}] Attempt {attempt + 1} failed: {e}. Retrying in {wait}s...")
            time.sleep(wait)
    else:
        print(f"[WARN] Skipping season {s} after multiple failures.")

if not all_dfs:
    raise RuntimeError("No data was fetched. Check your network or try again later.")

full = pd.concat(all_dfs, ignore_index=True)

# Convert season to a sortable start-year integer
full["SEASON_START"] = full["SEASON"].str.split("-").str[0].astype(int)

# For players who appear in multiple seasons, keep the **latest** name (most recent season)
full_sorted = full.sort_values(["personId", "SEASON_START"])
latest = full_sorted.groupby("personId", as_index=False)["playerFullName"].last()

output_path = "nba_players_2013_2025.csv"
latest.to_csv(output_path, index=False)
print(f"Saved {len(latest):,} unique players to {output_path}")
latest.head()
